# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

AI가 내놓은 결과만 보고 그것이 어떻게 나왔는지 모르면, 그 결과를 믿을 근거도 없다. AI도 실수를 하므로 결과는 점검해야 하고, 점검하려면 결과를 만든 파이썬 코드를 읽을 줄 알아야 한다.

이 노트북은 이 장에서 다룬 계산과 그림을 파이썬으로 직접 해 본다. 거래를 단지별로 모으고, 훈련 데이터와 시험 데이터로 나누고, 닮은 단지 모형과 회귀식으로 가격을 예측해 오차로 채점한다. 보고서의 표에 실린 숫자가 모두 여기서 나온다. 코드는 될 수 있는 한 단순하게 썼다.

1. 데이터 읽기 — 다중회귀분석 장과 같은 CSV를 읽는다
2. 단지별로 모으기 — 84㎡ 거래를 모아 한 행을 단지 하나로 만든다
3. 훈련 데이터와 시험 데이터 — 단지 500개를 무작위로 떼어 둔다
4. 회귀식으로 예측하기 — 식에 단지 한 곳을 넣어 예측값을 계산한다
5. 오차로 채점하기 — "보통 몇 % 빗나가나"를 계산하고, 변수를 하나씩 더해 본다(표 2-2)
6. 닮은 단지 모형 — 거리를 계산해 닮은 단지를 찾는다(표 4-1)
7. 훈련 오차와 시험 오차 — 0%와 16%(표 3-2)
8. 점을 모두 지나려는 선 — 거래 30건에 직선과 곡선을 긋는다(그림 4-1)
9. 닮은 단지를 몇 개 볼 것인가 — 표 4-2와 그림 4-2
10. 교차검증 — 시험 데이터를 보지 않고 개수를 고른다
11. 분할을 바꿔 다시 채점하기 — 표 5-1
12. 누출 — 거래 한 건을 한 행으로 두면 6.5%가 나온다
13. 다른 모형들 — 표 6-2의 여섯 모형

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. 앞의 네 줄은 다른 장과 같다. 이 장에서 새로 쓰는 것은 scikit-learn(코드에서는 `sklearn`)이다. 머신러닝에서 가장 널리 쓰는 파이썬 패키지이고, 데이터를 나누는 도구와 여러 모형이 들어 있다. AI에게 예측 모형을 만들어 달라고 하면 대개 이 패키지를 쓴 코드를 내놓는다. Anaconda와 Colab에 모두 들어 있다.

matplotlib는 기본 글꼴에 한글이 없어서 그림의 한글이 네모로 깨진다. 그래서 한글 글꼴을 지정한다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)을 쓴다. Colab에는 한글 글꼴이 없으므로 새 셀에서 `!apt-get -qq install fonts-nanum`을 실행하고, 런타임을 다시 시작한 뒤 `NanumGothic`을 지정한다.

In [ ]:
import numpy as np                       # 수치 계산
import pandas as pd                      # 표(데이터프레임) 다루기
import statsmodels.formula.api as smf    # 회귀분석
import matplotlib.pyplot as plt          # 그림

from sklearn.model_selection import train_test_split, KFold   # 데이터 나누기
from sklearn.neighbors import NearestNeighbors                # 가까운 이웃 찾기

# 그림의 한글 글꼴
# 맥은 'AppleGothic', Colab은 'NanumGothic'(설치 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호(-) 깨짐 방지

## 1. 데이터 읽기

데이터는 다중회귀분석 장과 같은 CSV다(서울 아파트 매매 1년치). 책 사이트의 장 첫 페이지에서 내려받을 수 있다. 국토교통부 실거래가 공개시스템에서 자기 지역이나 다른 기간의 데이터를 받는 방법은 단순회귀분석 장의 직접 해 보기 2, "데이터 수집"에 단계별로 적어 두었다.

- 내 컴퓨터의 주피터: CSV를 이 노트북 파일과 같은 폴더에 두고, 아래 코드의 `CSV =` 줄에 파일 이름을 그대로 적는다.
- Colab: 왼쪽의 폴더 아이콘을 누르고 업로드 버튼으로 CSV를 올린 뒤 `CSV = '/content/파일이름.csv'`로 적는다.

`pd.read_csv`의 옵션 셋은 앞 장과 같다. 한글 인코딩(`encoding='cp949'`), 표 위의 안내문 15줄(`skiprows=15`), 거래금액의 천 단위 쉼표(`thousands=','`)를 하나씩 맡는다.

In [ ]:
CSV = 'seoul-apt-sales-2025-10-2026-10.csv'   # 자기 파일 이름으로 바꾼다

df = pd.read_csv(CSV,
                 encoding='cp949',   # 한글 인코딩
                 skiprows=15,        # 표 위 안내문 15줄 건너뜀
                 thousands=',')      # 51,500 → 51500
print('서울 전체', len(df), '건')

## 2. 단지별로 모으기

1절의 "데이터"에서 한 일이다. 먼저 다중회귀분석 장과 똑같이 해제거래와 직거래를 빼고 변수를 만든다. 63,163건이 남는다.

그다음이 이 장에서 새로 하는 일이다.

- **전용 84㎡ 거래만 남긴다.** 전용면적이 84㎡ 이상 85.5㎡ 미만인 거래다. 19,987건이 남는다.
- **단지별로 모은다.** 이름이 같은 단지가 다른 동네에도 있으므로, 구와 동과 단지 이름을 이어 붙인 것을 단지의 이름표(`단지`)로 삼는다. `groupby('단지')`가 이름표가 같은 거래를 한데 묶고, `.agg(...)`가 묶음마다 값을 하나씩 계산한다. 단지의 가격은 84㎡ 거래가격의 중위값(`'median'`)이다.
- **최고층을 붙인다.** 그 단지에서 거래된 집 가운데 가장 높은 층이다. 84㎡ 거래만이 아니라 모든 면적의 거래(`d`)에서 찾는다.

이렇게 만든 표 `g`는 한 행이 단지 하나다. 2,709행이 나와야 한다. 이 장의 모든 계산은 이 표에서 출발한다.

In [ ]:
d = df[df['해제사유발생일'] == '-']                  # 해제거래 제외
d = d[d['거래유형'] == '중개거래'].copy()            # 직거래 제외

d['가격'] = d['거래금액(만원)'] / 10000              # 만원 → 억원
d['면적'] = d['전용면적(㎡)']
d['연식'] = 2026 - d['건축년도']                     # 지은 지 몇 년
addr = d['시군구'].str.split()                       # "서울특별시 노원구 상계동" → 낱말 셋
d['구'] = addr.str[1]
d['단지'] = addr.str[1] + ' ' + addr.str[2] + ' ' + d['단지명']   # 구 + 동 + 단지 이름
print('서울 매매', len(d), '건')

e = d[(d['면적'] >= 84) & (d['면적'] < 85.5)]        # 전용 84㎡ 거래만
print('전용 84㎡ 거래', len(e), '건')

g = e.groupby('단지').agg(구=('구', 'first'),         # 단지마다 한 행
                          연식=('연식', 'median'),
                          가격=('가격', 'median'),   # 단지의 가격 = 84㎡ 중위 거래가
                          거래=('가격', 'size'))     # 84㎡ 거래 건수
g['최고층'] = d.groupby('단지')['층'].max()          # 모든 면적의 거래에서 가장 높은 층
g = g.reset_index()
g['로그가격'] = np.log(g['가격'])

print('단지', len(g), '개')
print('단지의 가격: 최저', g['가격'].min(), '억원, 중위', g['가격'].median(), '억원, 최고', g['가격'].max(), '억원')
print('84㎡ 거래가 1건뿐인 단지', (g['거래'] == 1).sum(), '개')
g.head()

## 3. 훈련 데이터와 시험 데이터

1절의 "처음 보는 단지의 가격"에서 한 일이다. 2,709개 단지 가운데 500개를 무작위로 골라 시험 데이터로 떼어 둔다. 남은 2,209개가 훈련 데이터다.

`train_test_split`이 이 일을 한 줄로 해 준다.

- `test_size=500`: 시험 데이터로 뗄 행의 수. `test_size=0.2`처럼 소수로 적으면 전체의 20%를 뗀다.
- `random_state=SEED`: 무작위로 고르는 방법을 고정하는 번호(시드)다. 같은 번호를 주면 몇 번을 실행해도 같은 단지가 뽑힌다. 이 번호가 다르면 다른 단지가 뽑히고 점수도 조금 달라진다(11절). 이 장의 숫자는 모두 `20261116`으로 나눈 것이다.

이제부터 `tr`이 훈련 데이터, `te`가 시험 데이터다. 모형은 `tr`만으로 만들고, `te`의 가격은 채점할 때만 쓴다.

In [ ]:
SEED = 20261116                                   # 분할을 고정하는 번호

tr, te = train_test_split(g, test_size=500, random_state=SEED)
tr, te = tr.copy(), te.copy()
print('훈련 데이터', len(tr), '개 단지, 시험 데이터', len(te), '개 단지')

## 4. 회귀식으로 예측하기

2절의 "같은 회귀식, 다른 쓰임"이다. 보고서의 회귀식을 훈련 데이터로 추정한다.

로그(가격) = 최고층 + 연식 + 연식² + 구 더미 24개 (기준: 노원구)

회귀식을 쓰는 법은 다중회귀분석 장과 같다. `I(연식**2)`는 연식의 제곱이고, `C(구, Treatment('노원구'))`는 노원구를 기준으로 한 구 더미다. 달라진 것은 `data=tr`, 곧 훈련 데이터만 넣는다는 점이다.

계수를 하나하나 해석하지 않고 식에 숫자를 넣는다. 본문의 예는 시험 데이터에 있는 성북구 단지(지은 지 21년, 최고층 15층)였다. 계수를 꺼내 손으로 계산한 값과 `m.predict`가 계산한 값이 같은지 본다. 나온 값은 로그가격이므로 `np.exp`로 억원으로 되돌린다. 8.00억원이 나와야 한다.

In [ ]:
GU = "C(구, Treatment('노원구'))"                  # 구 더미 24개, 기준은 노원구
formula = '로그가격 ~ 최고층 + 연식 + I(연식**2) + ' + GU

m = smf.ols(formula, data=tr).fit()               # 훈련 데이터로만 추정
b = m.params
print('상수', round(b['Intercept'], 3), ' 최고층', round(b['최고층'], 4),
      ' 연식', round(b['연식'], 4), ' 연식²', round(b['I(연식 ** 2)'], 6),
      ' 성북구', round(b[GU + '[T.성북구]'], 3))

# 시험 데이터의 성북구 단지 한 곳 (21년, 15층)
one = te[(te['구'] == '성북구') & (te['연식'] == 21) & (te['최고층'] == 15)]
print(one[['구', '연식', '최고층', '가격']])

by_hand = (b['Intercept'] + b['최고층'] * 15 + b['연식'] * 21
           + b['I(연식 ** 2)'] * 21 ** 2 + b[GU + '[T.성북구]'])       # 식에 숫자를 넣는다
print('손으로 계산한 로그가격', round(by_hand, 3), '→', round(np.exp(by_hand), 2), '억원')
print('m.predict로 계산      ', round(np.exp(m.predict(one)).iloc[0], 2), '억원')

같은 계산을 훈련 데이터와 시험 데이터의 모든 단지에 한다. `m.predict(표)`는 표의 행마다 예측값을 계산한다. 예측한 가격을 `회귀식`이라는 열로 붙여 둔다.

In [ ]:
tr['회귀식'] = np.exp(m.predict(tr))      # 예측한 가격 (억원)
te['회귀식'] = np.exp(m.predict(te))
te[['구', '연식', '최고층', '가격', '회귀식']].head()

## 5. 오차로 채점하기

2절의 "한 단지씩 채점한다"와 "500개 단지의 점수를 숫자 하나로"다. 한 단지의 오차는 예측한 가격이 실제 가격에서 벗어난 크기를 실제 가격으로 나눈 것이다.

- `예측 / 실제 - 1`: 예측이 실제보다 몇 % 높은가(낮으면 음수)
- `np.abs(...)`: 높았는지 낮았는지를 가리지 않게 부호를 뗀다
- `np.median(...)`: 단지마다 계산한 오차의 중위값

이 세 가지를 묶어 `err`라는 함수로 만들어 둔다. 이 노트북의 채점은 모두 이 함수가 한다. $R^2$을 계산하는 함수 `r2`도 함께 만든다. 가격에 로그를 취한 값으로 계산한다(보고서 표의 주).

시험 데이터의 오차는 중위값 15.8%, 평균 19.9%가 나와야 한다.

In [ ]:
def err(actual, pred):
    # 보통 몇 % 빗나가나: 단지마다 계산한 오차의 중위값
    return np.median(np.abs(pred / actual - 1)) * 100

def r2(actual, pred):
    # R² (로그가격 기준): 평균으로 예측할 때의 흩어짐 가운데 줄인 몫
    y, p = np.log(actual), np.log(pred)
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

each = np.abs(te['회귀식'] / te['가격'] - 1) * 100          # 단지마다의 오차 (%)
print('중위값', round(each.median(), 1), '%  평균', round(each.mean(), 1), '%  최대', round(each.max()), '%')
print('5% 안쪽', round((each < 5).mean() * 100), '%  10% 안쪽', round((each < 10).mean() * 100),
      '%  50% 넘게', round((each > 50).mean() * 100), '%')
print('회귀식: 시험 오차', round(err(te['가격'], te['회귀식']), 1), '%  시험 R²', round(r2(te['가격'], te['회귀식']), 2))

**비교의 출발점** — 단지에 대해 아무것도 모를 때의 답은 평균 가격이다. 훈련 데이터의 로그가격 평균을 가격으로 되돌린 10.8억원을 시험 데이터 500개 단지 모두에 답하고 같은 함수로 채점한다. 36.4%가 나와야 한다.

**변수를 하나씩 더하기**(표 2-2) — 회귀식에 최고층, 연식, 구를 하나씩 더하며 다시 추정하고 시험 데이터로 채점한다. 구를 더할 때 오차가 가장 크게 줄어드는 것을 확인한다.

In [ ]:
base = np.exp(tr['로그가격'].mean())                        # 전체 평균 가격 하나
te['평균'] = base
print('전체 평균 가격', round(base, 1), '억원 → 시험 오차', round(err(te['가격'], te['평균']), 1), '%')

steps = [('최고층',           '로그가격 ~ 최고층'),
         ('최고층, 연식',     '로그가격 ~ 최고층 + 연식 + I(연식**2)'),
         ('최고층, 연식, 구', formula)]
rows = {'없음 (전체 평균 가격 하나로)': {'오차(%)': err(te['가격'], te['평균']), 'R²': r2(te['가격'], te['평균'])}}
for name, f in steps:
    p = np.exp(smf.ols(f, data=tr).fit().predict(te))      # 훈련 데이터로 추정, 시험 데이터로 채점
    rows[name] = {'오차(%)': err(te['가격'], p), 'R²': r2(te['가격'], p)}
pd.DataFrame(rows).T.round({'오차(%)': 1, 'R²': 2}) + 0     # + 0은 -0.00을 0.00으로

그림 2-1을 그린다. 왼쪽은 예측한 가격(가로)과 실제 가격(세로)이고, 점 하나가 단지 하나다. 실선은 예측과 실제가 같은 자리다. 두 축은 로그 눈금으로 그린다(`set_xscale('log')`). 오른쪽은 단지마다의 오차를 막대그림(`hist`)으로 그린 것이다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(te['회귀식'], te['가격'], s=10, alpha=0.5, color='#2a78d6')
ax[0].plot([3, 60], [3, 60], color='#eb6834')              # 예측 = 실제
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel('회귀식이 예측한 가격 (억원)')
ax[0].set_ylabel('실제 가격 (억원)')
ax[1].hist(each, bins=np.arange(0, 105, 5), color='#2a78d6')
ax[1].axvline(each.median(), color='#eb6834')              # 중위값
ax[1].set_xlabel('오차 (%)')
ax[1].set_ylabel('단지 수')
plt.show()

## 6. 닮은 단지 모형

3절의 "닮은 단지 모형 — 가장 가까운 이웃"이다. 닮은 단지는 같은 구 안에서 찾고, 얼마나 닮았는지는 연식의 차이와 최고층의 차이를 합친 거리로 따진다. 단위가 다른 두 변수를 합치려고 차이를 훈련 데이터의 표준편차로 나눈다.

두 변수를 숫자만 든 표로 꺼내는 일을 여러 번 하므로 함수 `feat`로 만들어 둔다. `.std(0)`이 열마다 표준편차를 계산한다. 최고층 7.2층, 연식 8.2년이 나와야 한다. 연식이 1년, 최고층이 1층 다른 두 단지의 거리는 0.18이다.

In [ ]:
def feat(t):
    # 닮음을 따지는 두 변수(최고층, 연식)를 숫자만 든 표로 꺼낸다
    return np.column_stack([t['최고층'], t['연식']])

sd = feat(tr).std(0)                               # 훈련 데이터의 표준편차 (최고층, 연식 순서)
print('표준편차: 최고층', round(sd[0], 1), '층, 연식', round(sd[1], 1), '년')
print('연식 1년, 최고층 1층 다른 두 단지의 거리', round(np.sqrt((1 / sd[1]) ** 2 + (1 / sd[0]) ** 2), 2))

**한 단지의 닮은 단지 찾기**(표 4-1) — 시험 데이터의 구로구 단지(33년, 15층, 실제 가격 6.15억원)와 닮은 단지를 훈련 데이터의 구로구 단지 113곳에서 찾는다.

1. 구로구의 훈련 단지 `a`와 시험 단지 `b`를 고른다.
2. 두 변수를 표준편차로 나눠 같은 눈금으로 바꾼다(`za`, `zb`). 평균을 빼는 것은 거리에 영향을 주지 않지만, 표준화할 때는 평균을 빼고 나누는 것이 관례여서 그렇게 했다.
3. `NearestNeighbors`가 `b`의 단지마다 `a`의 단지를 거리가 짧은 순서로 줄 세운다. `kneighbors`가 돌려주는 `order`는 닮은 순서대로 늘어놓은 `a`의 행 번호다. `n_neighbors=len(a)`로 구의 단지 전부를 줄 세워 달라고 했다.

닮은 순서대로 10곳을 뽑아 거리와 함께 본다. 첫째가 11.75억원이고, 연식이 1년 다른 셋째는 6.00억원이다. 1개, 10개, 30개, 구 전체를 평균한 답도 계산한다. 평균은 로그가격으로 내고 가격으로 되돌린다(11.75억, 6.5억, 6.9억, 7.6억원).

연식과 최고층이 똑같아 거리가 같은 단지가 둘 이상이면(아래 표의 둘째와 셋째) 어느 쪽이 먼저 놓일지는 `NearestNeighbors`가 정한다. 그래서 패키지의 판이 다르면 순서가 바뀌어 뒤의 점수가 소수 첫째 자리에서 조금 달라질 수 있다.

In [ ]:
mu = feat(tr).mean(0)                              # 훈련 데이터의 평균

a = tr[tr['구'] == '구로구']                        # 구로구의 훈련 단지
b = te[te['구'] == '구로구']                        # 구로구의 시험 단지
za = (feat(a) - mu) / sd                           # 같은 눈금으로
zb = (feat(b) - mu) / sd

order = NearestNeighbors(n_neighbors=len(a), algorithm='brute').fit(za).kneighbors(zb, return_distance=False)

i = np.where((b['연식'] == 33) & (b['최고층'] == 15))[0][0]    # 33년, 15층 단지가 b의 몇 번째 행인가
print(b.iloc[i][['구', '연식', '최고층', '가격']].to_dict(), '/ 구로구의 훈련 단지', len(a), '곳')

near = a.iloc[order[i]]                            # 닮은 순서대로 늘어놓은 구로구의 훈련 단지
dist = np.sqrt((((feat(near) - feat(b)[i]) / sd) ** 2).sum(1))    # 거리의 식
print(near[['연식', '최고층', '가격']].assign(거리=dist.round(2)).head(10).to_string(index=False))

for k in [1, 10, 30, len(a)]:
    print(k, '개의 평균', round(np.exp(near['로그가격'].head(k).mean()), 2), '억원')

**함수로 묶기** — 같은 일을 모든 구, 모든 단지에 하도록 함수 `knn`으로 묶는다. `train`의 단지에서 닮은 단지를 찾아 `target`의 단지마다 가격을 예측한다. `ks`에는 닮은 단지를 몇 개 볼지를 여러 개 적는다. 구마다 닮은 순서를 한 번만 구해 두고, 앞에서 $k$개씩 잘라 평균한다(`near[:, :k]`). 돌려주는 표는 행이 `target`의 단지, 열이 $k$이고, 값은 예측한 가격이다.

구의 훈련 단지가 $k$개보다 적으면 그 구의 단지 전부를 평균한다(표 4-2의 주). 구마다 훈련 단지가 200개보다 적으므로, $k$ = 200은 "같은 구의 모든 단지"와 같다.

표준편차와 평균은 함수 안에서 `train`으로 다시 계산한다. 10절에서 훈련 데이터를 다시 나눠 쓸 때도 그때의 `train`만 보게 하려는 것이다.

In [ ]:
def knn(train, target, ks):
    # target의 단지마다, 같은 구의 train 단지 가운데 닮은 k개의 평균으로 가격을 예측한다
    mu, sd = feat(train).mean(0), feat(train).std(0)             # train의 평균과 표준편차
    pred = pd.DataFrame(index=target.index, columns=ks, dtype=float)
    for gu in target['구'].unique():                             # 구마다
        a = train[train['구'] == gu]                             # 닮은 단지를 찾을 곳
        b = target[target['구'] == gu]                           # 가격을 예측할 단지
        za = (feat(a) - mu) / sd
        zb = (feat(b) - mu) / sd
        n = min(max(ks), len(a))                                 # 줄 세울 단지 수
        order = NearestNeighbors(n_neighbors=n, algorithm='brute').fit(za).kneighbors(zb, return_distance=False)
        near = a['로그가격'].to_numpy()[order]                    # 닮은 순서대로 늘어놓은 로그가격
        for k in ks:
            pred.loc[b.index, k] = np.exp(near[:, :k].mean(1))   # 앞에서 k개의 평균 → 억원
    return pred

KS = [1, 2, 3, 5, 10, 20, 30, 50, 100, 200]    # 200 = 같은 구의 모든 단지
p_tr = knn(tr, tr, KS)                         # 훈련 데이터의 단지를 예측
p_te = knn(tr, te, KS)                         # 시험 데이터의 단지를 예측
p_te.head().round(2)

## 7. 훈련 오차와 시험 오차

3절의 "닮은 단지 1개 모형은 이렇게 맞힌다"와 표 3-2다. 닮은 단지를 1개만 보는 모형을 두 가지 데이터로 채점한다.

- 훈련 데이터의 단지를 예측하면(`p_tr[1]`) 가장 닮은 단지로 자기 자신이 뽑힌다. 예측이 실제 가격과 똑같은 단지가 1,926개이고, 오차의 중위값은 0.0%다.
- 시험 데이터의 단지를 예측하면(`p_te[1]`) 자기 자신이 뽑힐 수 없다. 오차는 16.1%다.

회귀식은 두 점수가 거의 같다(둘 다 15.8%). `np.isclose(a, b)`는 두 값이 같은지 본다. 소수 계산에는 아주 작은 오차가 끼므로 `==` 대신 쓴다.

In [ ]:
same = np.isclose(p_tr[1], tr['가격'])             # 예측이 실제 가격과 똑같은 훈련 단지
print('예측 = 실제인 훈련 단지', same.sum(), '개, 나머지', (~same).sum(), '개')

table = pd.DataFrame({
    '훈련 오차(%)': [err(tr['가격'], p_tr[1]), err(tr['가격'], tr['회귀식'])],
    '시험 오차(%)': [err(te['가격'], p_te[1]), err(te['가격'], te['회귀식'])],
    '훈련 R²':      [r2(tr['가격'], p_tr[1]),  r2(tr['가격'], tr['회귀식'])],
    '시험 R²':      [r2(te['가격'], p_te[1]),  r2(te['가격'], te['회귀식'])]},
    index=['닮은 단지 1개', '회귀식'])
print(table.round({'훈련 오차(%)': 1, '시험 오차(%)': 1, '훈련 R²': 2, '시험 R²': 2}))

big = lambda p: round((np.abs(p / te['가격'] - 1) > 0.5).mean() * 100)   # 오차가 50%를 넘는 시험 단지의 비율
print('오차 50% 넘는 시험 단지: 회귀식', big(te['회귀식']), '%, 닮은 단지 1개', big(p_te[1]), '%')

## 8. 점을 모두 지나려는 선

4절의 그림 4-1이다. 단지가 아니라 단순회귀분석 장의 노원구 거래(전용면적과 거래가격)를 쓴다. 전용 30 ~ 135㎡ 거래 7,016건에서 30건만 무작위로 뽑아 선을 긋고, 나머지 거래를 새로운 거래로 삼아 채점한다.

- `np.random.default_rng(1)`은 시드 1로 난수 생성기를 만들고, `.choice(전체 건수, 30, replace=False)`가 겹치지 않게 30건의 행 번호를 뽑는다.
- `np.polyfit(x, y, 차수)`는 점들에 가장 가까운 다항식을 찾는다. 1차는 직선, 3차는 완만한 곡선이고, 12차는 열두 번까지 휘어질 수 있는 구불구불한 곡선이다. 차수가 선의 유연함을 정하는 다이얼이다.
- `np.polyval(계수, x)`는 그 다항식에 `x`를 넣어 값을 계산한다.
- 면적은 평균을 빼고 표준편차로 나눠서 넣는다(`(x - x0) / s0`). 84의 12제곱 같은 큰 수가 생기면 계산이 불안정해지기 때문이다.

30건의 오차는 차수가 높을수록 줄어들고(15%, 11%, 7%), 새로운 거래의 오차는 거꾸로 커진다(16%, 18%, 28%).

In [ ]:
nw = d[(d['구'] == '노원구') & (d['면적'] >= 30) & (d['면적'] <= 135)]
x, y = nw['면적'].to_numpy(), nw['가격'].to_numpy()
print('노원구 전용 30 ~ 135㎡ 거래', len(x), '건')

pick = np.random.default_rng(1).choice(len(x), 30, replace=False)   # 30건을 뽑는다
rest = np.setdiff1d(np.arange(len(x)), pick)                        # 나머지 거래
x0, s0 = x[pick].mean(), x[pick].std()

fig, ax = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
xs = np.linspace(x[pick].min(), x[pick].max(), 300)
for i, (deg, name) in enumerate([(1, '직선'), (3, '완만한 곡선'), (12, '구불구불한 곡선')]):
    co = np.polyfit((x[pick] - x0) / s0, y[pick], deg)              # 30건에 선을 긋는다
    f = lambda v: np.polyval(co, (v - x0) / s0)                     # 면적 → 예측 가격
    e_in, e_new = err(y[pick], f(x[pick])), err(y[rest], f(x[rest]))
    print(name, ' 30건', round(e_in, 1), '%  새로운 거래', round(e_new, 1), '%')
    ax[i].scatter(x[rest], y[rest], s=3, color='lightgray')
    ax[i].scatter(x[pick], y[pick], s=25, color='#2a78d6')
    ax[i].plot(xs, f(xs), color='#eb6834', linewidth=2)
    ax[i].set_title(f'{name}: 30건 {e_in:.0f}%, 새로운 거래 {e_new:.0f}%')
    ax[i].set_xlim(35, 90)
    ax[i].set_ylim(0, 18)
    ax[i].set_xlabel('전용면적 (㎡)')
ax[0].set_ylabel('거래가격 (억원)')
plt.show()

## 9. 닮은 단지를 몇 개 볼 것인가

4절의 표 4-2와 그림 4-2다. 6절에서 만들어 둔 `p_tr`, `p_te`에는 닮은 단지를 1개부터 구 전체(200)까지 볼 때의 예측이 열마다 들어 있다. 열마다 채점해 표로 만든다.

훈련 데이터의 점수는 적게 볼수록 좋아지기만 하고, 시험 데이터의 점수는 가운데가 가장 좋은 것을 확인한다. 보고서의 그림 오른쪽 패널이 이 표의 $R^2$ 두 열을 그린 것이다. 가로축은 1, 10, 100이 같은 간격이 되도록 로그 눈금으로 그린다.

In [ ]:
score = pd.DataFrame({'훈련 오차(%)': [err(tr['가격'], p_tr[k]) for k in KS],
                      '시험 오차(%)': [err(te['가격'], p_te[k]) for k in KS],
                      '훈련 R²':      [r2(tr['가격'], p_tr[k]) for k in KS],
                      '시험 R²':      [r2(te['가격'], p_te[k]) for k in KS]}, index=KS)
score.index.name = '닮은 단지'
print(score.round({'훈련 오차(%)': 1, '시험 오차(%)': 1, '훈련 R²': 2, '시험 R²': 2}))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, name in enumerate(['오차(%)', 'R²']):
    ax[i].plot(KS, score['훈련 ' + name], marker='o', color='#2a78d6', label='훈련 데이터')
    ax[i].plot(KS, score['시험 ' + name], marker='o', color='#eb6834', label='시험 데이터')
    ax[i].set_xscale('log')
    ax[i].set_xticks([1, 10, 100, 200])
    ax[i].set_xticklabels(['1개', '10개', '100개', '구 전체'])
    ax[i].set_xlabel('닮은 단지를 몇 개 보나')
    ax[i].set_ylabel(name)
    ax[i].legend()
ax[1].axhline(r2(te['가격'], te['회귀식']), color='gray', linestyle='--')   # 회귀식의 시험 R²
plt.show()

## 10. 교차검증

4절의 "몇 개를 볼지는 어떻게 골랐나"다. 9절의 표에서 시험 데이터의 점수가 가장 좋은 개수를 고르면 시험 데이터를 보고 모형을 만든 셈이 된다. 그래서 훈련 데이터 안에서 고른다.

`KFold(5, shuffle=True, random_state=SEED)`는 행을 무작위로 섞어 다섯 조각으로 나누는 도구다. `.split(tr)`은 다섯 번에 걸쳐 "모형을 만드는 네 조각의 행 번호"와 "검증에 쓰는 한 조각의 행 번호"를 차례로 내준다(그림 4-3). 그때마다 네 조각(`fit`)에서 닮은 단지를 찾아 한 조각(`val`)의 가격을 예측하고 채점한다. 다섯 점수의 평균이 교차검증의 점수다. 이 셀에는 `te`가 한 번도 나오지 않는다.

교차검증의 오차는 1개 17.0%, 3개 15.1%, 5개 15.0%, 10개 15.1%, 구 전체 22.1%가 나와야 한다. 5개와 10개는 거의 같다. 보고서는 10개를 썼다. 개수를 10개로 정한 뒤에 시험 데이터로 채점한 점수가 보고서의 14.6%다.

In [ ]:
folds = KFold(5, shuffle=True, random_state=SEED)

scores = []
for i_fit, i_val in folds.split(tr):               # 다섯 번
    fit, val = tr.iloc[i_fit], tr.iloc[i_val]      # 모형을 만드는 네 조각, 검증 데이터 한 조각
    p = knn(fit, val, KS)
    scores.append([err(val['가격'], p[k]) for k in KS])

cv = pd.Series(np.mean(scores, axis=0), index=KS)  # 다섯 점수의 평균
print('교차검증의 오차(%)')
print(cv.round(1).to_string())

print('10개로 정한 뒤 시험 데이터로 한 번 채점:', round(err(te['가격'], p_te[10]), 1), '%')

## 11. 분할을 바꿔 다시 채점하기

5절의 표 5-1이다. 3절부터 한 일(나누기, 회귀식 추정, 닮은 단지 찾기, 채점)을 함수 `run`으로 묶고, `random_state`만 바꿔 여섯 번 되풀이한다. 첫 줄이 보고서의 분할이고, 나머지 다섯 줄이 보고서 표의 맨 오른쪽 열 "다르게 나눈 다섯 번의 시험 오차"다.

같은 모형인데도 분할에 따라 점수가 1 ~ 2%포인트 달라지고, 다른 분할 1에서는 회귀식이 닮은 단지 10개 모형보다 낮은 것을 확인한다.

In [ ]:
def run(seed):
    a, b = train_test_split(g, test_size=500, random_state=seed)   # 다르게 나눈다
    reg = np.exp(smf.ols(formula, data=a).fit().predict(b))        # 회귀식
    p = knn(a, b, [1, 10, 200])                                    # 닮은 단지 1개, 10개, 구 전체
    flat = np.full(len(b), np.exp(a['로그가격'].mean()))            # 전체 평균 가격 하나
    return {'전체 평균': err(b['가격'], flat), '회귀식': err(b['가격'], reg),
            '닮은 단지 1개': err(b['가격'], p[1]), '닮은 단지 10개': err(b['가격'], p[10]),
            '같은 구의 모든 단지': err(b['가격'], p[200])}

seeds = pd.DataFrame({'보고서의 분할': run(SEED)} | {f'다른 분할 {s}': run(s) for s in range(1, 6)}).T
print(seeds.round(1))
print(seeds.iloc[1:].agg(['min', 'max']).round(1))     # 다르게 나눈 다섯 번의 범위

## 12. 누출

5절의 "시험지에 기출문제가 섞였다면"이다. 단지로 모으지 않고 84㎡ 거래 19,987건을 한 건 = 한 행으로 둔다. 변수는 같다(구, 연식, 최고층). 거래마다 그 단지의 최고층을 붙이려고 `merge`로 2절의 표 `g`와 단지 이름표를 맞춘다.

**거래를 무작위로 나누면** — 20%를 시험 데이터로 떼고, 6절의 함수 `knn`을 그대로 써서 가장 닮은 거래 1건의 가격으로 예측한다. 오차가 6.5%로 나온다. `isin`으로 세어 보면 시험 거래의 96%는 같은 단지의 다른 거래가 훈련 데이터에 있다.

**단지째 나누면** — `GroupShuffleSplit`은 `groups`로 준 이름표가 같은 행을 훈련 데이터와 시험 데이터 가운데 한쪽에만 넣는다. 단지 이름표를 주면 같은 단지의 거래가 양쪽에 함께 들어가지 않는다. 오차는 15.5%로 돌아온다.

회귀식은 무작위로 나눠도 15.4%다. 누출로 점수가 부풀려지는 것은 외우는 힘이 센 모형이다.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

t = e.merge(g[['단지', '최고층']], on='단지')        # 거래 한 건 = 한 행, 단지의 최고층을 붙인다
t['로그가격'] = np.log(t['가격'])
print('84㎡ 거래', len(t), '건')

# ① 거래를 무작위로 나눈다
a, b = train_test_split(t, test_size=0.2, random_state=SEED)
p = knn(a, b, [1, 10])
reg = np.exp(smf.ols(formula, data=a).fit().predict(b))
print('무작위로: 가장 닮은 거래 1건', round(err(b['가격'], p[1]), 1), '%  회귀식', round(err(b['가격'], reg), 1), '%')
print('  같은 단지의 거래가 훈련 데이터에 있는 시험 거래', round(b['단지'].isin(a['단지']).mean() * 100), '%')

# ② 단지째 나눈다
i_a, i_b = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED % 1000).split(t, groups=t['단지']))
a, b = t.iloc[i_a], t.iloc[i_b]
p = knn(a, b, [1, 10])
print('단지째:   가장 닮은 거래 1건', round(err(b['가격'], p[1]), 1), '%')
print('  같은 단지의 거래가 훈련 데이터에 있는 시험 거래', round(b['단지'].isin(a['단지']).mean() * 100), '%')

## 13. 다른 모형들

6절의 표 6-2와 그림 6-1이다. 같은 훈련 데이터와 같은 변수(구, 연식, 최고층)로 다른 모형들을 만들고, 같은 시험 데이터로 채점한다. 모형은 모두 scikit-learn에 들어 있고 쓰는 법이 같다.

- `모형.fit(X, y)`: 훈련 데이터로 모형을 만든다
- `모형.predict(X)`: 예측값을 계산한다

scikit-learn의 모형은 회귀식을 글자로 받지 않고 숫자로 된 표 `X`를 받는다. 그래서 구 이름을 `pd.get_dummies`로 0과 1의 스위치로 바꿔 넣는다(다중회귀분석 장의 더미 변수). 맞히는 것 `y`는 로그가격이다.

**의사결정나무** — `max_leaf_nodes`가 묶음의 수, 곧 나무의 다이얼이다. 먼저 묶음 네 개짜리 작은 나무를 만들어 `export_text`로 질문을 글자로 찍어 본다. 그림 6-1의 나무다. `연식 <= 14.50`은 "지은 지 14년 이하인가", `강남구 <= 0.50`은 "강남구 스위치가 꺼져 있는가", 곧 "강남구가 아닌가"라는 질문이다. `value`는 그 묶음의 평균 로그가격이어서 `np.exp`로 되돌려 읽는다(2.82 → 16.7억원).

`max_leaf_nodes`를 주지 않으면 나무는 끝까지 가른다. 묶음이 1,921개가 되고, 훈련 오차 0.0%, 시험 오차 17.3%가 나온다. 7절의 닮은 단지 1개 모형과 같은 모습이다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text

X = pd.get_dummies(g[['최고층', '연식', '구']], columns=['구'], prefix='', prefix_sep='', dtype=float)   # 최고층, 연식, 구 스위치 25개
X_tr, X_te = X.loc[tr.index], X.loc[te.index]       # 3절과 같은 분할
y_tr = tr['로그가격']

def test(model, A=X_tr, B=X_te):
    # 모형의 훈련 오차와 시험 오차 (%)
    return [float(round(err(tr['가격'], np.exp(model.predict(A))), 1)), float(round(err(te['가격'], np.exp(model.predict(B))), 1))]

small = DecisionTreeRegressor(max_leaf_nodes=4, random_state=0).fit(X_tr, y_tr)   # 묶음 네 개
print(export_text(small, feature_names=list(X.columns)))

full = DecisionTreeRegressor(random_state=SEED).fit(X_tr, y_tr)                   # 끝까지 가른 나무
print('끝까지 가른 나무: 묶음', full.get_n_leaves(), '개, [훈련 오차, 시험 오차] =', test(full))

**다이얼 눈금을 교차검증으로 고르기** — 10절에서는 교차검증을 손으로 짰다. scikit-learn의 모형이라면 `GridSearchCV`가 대신 해 준다. 후보 눈금(`max_leaf_nodes`의 값 아홉 가지)과 조각을 나누는 방법(`cv=folds`, 10절의 다섯 조각)을 주면, 후보마다 교차검증의 점수를 구해 가장 좋은 눈금을 고르고, 그 눈금으로 훈련 데이터 전부를 써서 모형을 다시 만든다. 점수는 $R^2$로 매기게 했다(`scoring='r2'`). 묶음 80개가 뽑히고 시험 오차는 15.7%다.

**나머지 모형** — 랜덤 포레스트(`RandomForestRegressor`, 나무 300그루), 부스팅(`HistGradientBoostingRegressor`), 규제를 건 회귀식(`RidgeCV`)을 차례로 만든다. 랜덤 포레스트와 부스팅의 다이얼 눈금(`min_samples_leaf=3`, `max_iter=400`, `max_leaf_nodes=4`)은 같은 방법으로 교차검증해 고른 값을 적어 넣었다. 직접 고르게 하면 몇 분이 걸리기 때문이다. `RidgeCV`는 벌점의 세기(`alphas`)를 교차검증으로 고르는 일까지 스스로 한다. 벌점은 계수의 크기에 매기므로 변수의 눈금을 먼저 맞춘다(`StandardScaler`). `make_pipeline`은 이 두 단계를 하나로 잇는다. 회귀식 계열에는 연식의 제곱을 넣고 구 스위치 하나를 뺀 표 `X2`를 쓴다.

여섯 모형의 시험 오차가 14.0%에서 15.9% 사이에 모여 있는 것을 확인한다(표 6-2).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 의사결정나무: 묶음의 수를 교차검증으로 고른다
grid = {'max_leaf_nodes': [5, 10, 20, 30, 50, 80, 120, 200, 400]}
tree = GridSearchCV(DecisionTreeRegressor(random_state=SEED), grid, cv=folds, scoring='r2').fit(X_tr, y_tr)
print('교차검증이 고른 눈금', tree.best_params_)

# 랜덤 포레스트, 부스팅: 교차검증으로 고른 눈금을 적어 넣었다
forest = RandomForestRegressor(300, min_samples_leaf=3, random_state=SEED).fit(X_tr, y_tr)
boost = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1, max_leaf_nodes=4, random_state=SEED).fit(X_tr, y_tr)

# 규제를 건 회귀식(릿지): 최고층, 연식, 연식², 구 스위치 24개
X2 = X.drop(columns='강남구')
X2.insert(2, '연식제곱', X2['연식'] ** 2)
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25), cv=folds)).fit(X2.loc[tr.index], y_tr)

pd.DataFrame({'회귀식':                [round(err(tr['가격'], tr['회귀식']), 1), round(err(te['가격'], te['회귀식']), 1)],
              '규제를 건 회귀식':      test(ridge, X2.loc[tr.index], X2.loc[te.index]),
              '닮은 단지 10개':        [round(err(tr['가격'], p_tr[10]), 1), round(err(te['가격'], p_te[10]), 1)],
              '의사결정나무':          test(tree),
              '랜덤 포레스트':         test(forest),
              '부스팅':                test(boost)}, index=['훈련 오차(%)', '시험 오차(%)']).T

## 정리

이 노트북의 코드는 이 장에서 본 숫자를 직접 만들어 냈다. 보고서의 표는 세 가지에서 나온다. 단지를 훈련 데이터와 시험 데이터로 나누는 한 줄(3절), 예측을 내놓는 두 모형(4절의 회귀식, 6절의 함수 `knn`), 그리고 채점하는 함수 `err`(5절)다. 나머지 절은 이 셋을 되풀이해 쓴 것이다. 무엇으로 채점하느냐에 따라 같은 모형의 점수가 0%도 16%도 되고(7절), 행을 무엇으로 두고 어떻게 나누느냐에 따라 6.5%도 15.5%도 된다(12절).

자기 데이터로 분석하려면 1절의 `CSV` 파일 이름을 바꾸고, 3절의 `test_size`를 단지 수의 20%쯤으로 바꾸고, 4절의 기준 구(`'노원구'`)를 그 데이터에 있는 지역 이름으로 바꾼다. 4절의 성북구 단지, 6절의 구로구 단지, 8절의 노원구처럼 서울의 구 이름을 직접 쓴 셀은 건너뛰거나 자기 지역의 이름으로 바꾼다. 13절에서 구 스위치 하나를 빼는 줄(`'강남구'`)도 자기 지역의 이름으로 바꾼다. 단지가 적은 지역에서는 구마다 훈련 단지가 몇 개 안 되어 닮은 단지 모형의 점수가 크게 흔들린다. 11절처럼 분할을 바꿔 가며 점수가 얼마나 움직이는지 함께 본다.

AI가 만든 예측 보고서를 받았다면, 이 코드로 같은 방식의 점수가 나오는지 확인해 보는 것이 가장 확실한 점검이다. 분할이 다르면 숫자가 똑같지는 않다. 11절의 범위 안에 들어오는지를 본다.